### Classification using the Dataloader

#### Problem Statement 
- predict if a customer will exit or not based on different parameters

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [2]:
# Load the Data
df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### EDA

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [4]:
### Remove the unwanted featuers 
df.drop(['RowNumber','CustomerId','Surname'],axis =1,inplace=True)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


### Data Prepration

In [6]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(df['Geography'])
df['Geography'] = encoder.transform(df['Geography'])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0,Female,42,2,0.00,1,1,1,101348.88,1
1,608,2,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,0,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,0,Female,39,1,0.00,2,0,0,93826.63,0
4,850,2,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
# Convert gender to numric
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  int64  
 2   Gender           10000 non-null  int64  
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9)
memory usage: 859.5 KB


In [9]:
df.describe()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,650.528800,0.746300,0.545700,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,96.653299,0.827529,0.497932,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,350.000000,0.000000,0.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,584.000000,0.000000,0.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,652.000000,0.000000,1.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,718.000000,1.000000,1.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,850.000000,2.000000,1.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


### Split the Dataset

In [10]:
x = df.drop('Exited',axis=1).values
y = df['Exited'].values

In [11]:
# Split the Data into the Train And Test
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.7,random_state=42)


### Deep learning

In [12]:
# Find the deviece if any
device = 'cpu'

if torch.backends.mps.is_available():
    device='mps' #mps (metal performance sharder)
elif torch.cuda.is_available():
    device = 'cuda' # cuda (compute unified device architecture)
else:
    device='cpu'

print(f" founded device : {device} ")

 founded device : cuda 


In [13]:
# Safely convert to float32 tensors (re-run friendly)
x_train = torch.as_tensor(x_train, dtype=torch.float32).to(device)
x_test = torch.as_tensor(x_test, dtype=torch.float32).to(device)
y_train = torch.as_tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test = torch.as_tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

In [14]:
y_train.shape,x_train.shape

(torch.Size([7000, 1]), torch.Size([7000, 10]))

#### Create the tensor Datasets

In [15]:
from torch.utils.data import TensorDataset
# Create the Training Dataset
dataset_train =  TensorDataset(x_train,y_train)
# Create the testeing Dataset
dataset_test = TensorDataset(x_test,y_test)


### Create the Loaders

In [16]:
from torch.utils.data import DataLoader
loader_train = DataLoader(dataset_train,batch_size=64,shuffle=True)
loader_test = DataLoader(dataset_test,batch_size=64,shuffle=False)

In [17]:
# Create the Model
# Sequantial is container to contain all the fully connected layers
# all the neurons from previous layer will be connected to all neurons of next layer
model = torch.nn.Sequential(
    # input layer connecting to hidden layer with 8 neurons
    torch.nn.Linear(in_features=10,out_features=16),
    # SET the activation function on the first hidden layer as relu
    torch.nn.ReLU(),
    # Add Another Hidden Layer 4 neurons
    torch.nn.Linear(in_features=16,out_features=8),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=8,out_features=1),
    torch.nn.Sigmoid()
)
model = model.to(device)

In [18]:
print(model)

Sequential(
  (0): Linear(in_features=10, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)


In [19]:
# Set the hyperparameters 
# number of epochs
epochs = 1000
#learning Rate 
lr=0.01
#loss Function
loss_function = torch.nn.BCELoss()

# Set the optimizer
optimizer  = torch.optim.Adam(model.parameters(),lr=lr)

In [20]:
print(len(loader_train))

110


In [ ]:
losses = [] #Collect all losses

accuracy_metrics = []  #collect all accuracyes

for epoch in range(epochs):

    #Enable the train mode
    model.train()

    # get the training loss
    running_loss = 0

    for x_batch,y_batch in loader_train:
        # Predict the answers for the x_train
        # The Entire data can load in the memory in the at once so for bigger data it crashesh
        # so use Data loader for it
        y_pred = model(x_batch)

        # Reset the Gradients
        optimizer.zero_grad()

        # Calculate the loss
        loss = loss_function(y_pred,y_batch)

        # Cal the loss Gradients 
        loss.backward()

        #Update the Weights
        optimizer.step()

        running_loss+=loss.item() 
    
    # calculate the entire loss (total loss for all batches/total no.of batches)
    loss = running_loss/len(loader_train)
    
    #collect the losses : 
    losses.append(loss)
    if (epoch+1) %100==0 :
        print(f" {epoch+1} loss : {loss} ") #weight : {model.parameters[0]}  bais : {model.parameters[1]}")

 100 loss : 79.19507578069513 


### Model Evaluation

In [ ]:
# Enables the evaluation Mode : 
model.eval() 
# Desable the Training Prameters 
with torch.no_grad():
    for x_batch,y_batch in loader_test:
        y_pred = model(x_batch)
        prediction_classes = (y_pred >= 0.5).float()  
        accuracy = (prediction_classes==y_batch).float().mean() # This Opreation done on the GPU
print(accuracy)